In [16]:
import pandas as pd
import pyodbc
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "iframe_connected"

print("Imports OK")

Imports OK


In [17]:
fig_teste = go.Figure(go.Indicator(mode="number", value=123))
fig_teste.show()

In [18]:
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=SANKHYA_MOCKUP;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

print("Conexão OK")

Conexão OK


In [19]:
teste = pd.read_sql("""
SELECT TOP 10 *
FROM dbo.AD_PCPCEN;
""", conn)

display(teste)
print("Linhas retornadas:", len(teste))

,SEQUENCIA,CODEMP,NUOP,CODATIVIDADE,CODETAPA,CODMQP,DH_I,DH_F,TIRAGEM,SALDO_TIRAGEM,...,DHINC,USUINC,DHALT,USUALT,ADDEXCEDENTE,TMPPREV,ALTSUBSIMPR,TEMPOTOTAL,DTFIXADA,GERARREL
0,124576,2,57171,30006,2,2,None,None,1134.43,0.43,...,2026-03-02 08:46:07.190,775,2026-03-07 11:31:31.907,0,None,None,N1,101,N1,S1
1,124577,2,57171,30009,4,39,None,None,814.80,271.80,...,2026-03-02 08:46:07.190,775,2026-03-14 08:33:24.513,0,None,None,NaN,20,NaN,NaN
2,124578,2,57171,30045,5,47,None,None,814.80,813.80,...,2026-03-02 08:46:07.190,775,2026-03-16 06:40:46.947,0,None,None,NaN,10,NaN,NaN
3,124579,2,57171,30030,8,45,None,None,935.02,-8384.98,...,2026-03-02 08:46:07.190,775,2026-03-06 12:08:09.883,0,None,None,S1,45,N1,S1
4,124580,2,57171,49394,9,45,None,None,935.02,935.02,...,2026-03-02 08:46:07.190,775,2026-03-03 17:19:56.617,85,None,None,S1,43,N1,S1
5,124586,2,57173,30006,2,2,None,None,850.83,0.83,...,2026-03-02 08:52:30.117,775,2026-03-07 12:26:47.430,130,None,None,N1,59,N1,S1
6,124587,2,57173,30009,4,39,None,None,611.10,221.10,...,2026-03-02 08:52:30.117,775,2026-03-14 09:35:45.790,0,None,None,NaN,18,NaN,NaN
7,124588,2,57173,30045,5,47,None,None,611.10,610.10,...,2026-03-02 08:52:30.117,775,2026-03-16 06:38:00.120,0,None,None,NaN,10,NaN,NaN
8,124589,2,57173,30030,8,45,None,None,701.27,-6339.73,...,2026-03-02 08:52:30.117,775,2026-03-06 12:26:09.280,0,None,None,S1,42,N1,S1
9,124590,2,57173,49394,9,45,None,None,701.27,701.27,...,2026-03-02 08:52:30.117,775,2026-03-03 17:20:01.230,85,None,None,n1,42,N1,S1


Linhas retornadas: 10


In [22]:
datas = pd.read_sql("""
SELECT 
    MIN(DT_ENTREGA) AS MENOR_DATA,
    MAX(DT_ENTREGA) AS MAIOR_DATA,
    COUNT(*) AS TOTAL_LINHAS
FROM dbo.AD_PCPCEN;
""", conn)

display(datas)

,MENOR_DATA,MAIOR_DATA,TOTAL_LINHAS
0,2026-03-05,2026-07-31,1387


In [23]:
data_inicio = "20260501"
data_fim = "20260601"

print("Período definido:", data_inicio, "até", data_fim)

Período definido: 20260501 até 20260601


In [24]:
query_kpis = f"""
SELECT
    SUM(ISNULL(TIRAGEM, 0)) AS TIRAGEM_TOTAL,
    COUNT(DISTINCT NUOP) AS TOTAL_OPS,

    SUM(CASE WHEN CODETAPA = 2 AND UPPER(ISNULL(TIPOIMPRESSAO, '')) LIKE 'I%' THEN 1 ELSE 0 END) AS IMPRESSOES_INTERNAS,

    SUM(CASE WHEN CODETAPA = 2 AND UPPER(ISNULL(TIPOIMPRESSAO, '')) LIKE 'E%' THEN 1 ELSE 0 END) AS IMPRESSOES_EXTERNAS,

    COUNT(DISTINCT NUOP)
    -
    (
        SUM(CASE WHEN CODETAPA = 2 AND UPPER(ISNULL(TIPOIMPRESSAO, '')) LIKE 'I%' THEN 1 ELSE 0 END)
        +
        SUM(CASE WHEN CODETAPA = 2 AND UPPER(ISNULL(TIPOIMPRESSAO, '')) LIKE 'E%' THEN 1 ELSE 0 END)
    ) AS OPS_MONOCAMADA,

    CAST(
        SUM(ISNULL(TIRAGEM, 0)) / NULLIF(COUNT(DISTINCT NUOP), 0)
        AS DECIMAL(18,2)
    ) AS TIRAGEM_MEDIA_OP

FROM dbo.AD_PCPCEN
WHERE DT_ENTREGA >= CONVERT(datetime, '{data_inicio}', 112)
  AND DT_ENTREGA <  CONVERT(datetime, '{data_fim}', 112);
"""

kpis = pd.read_sql(query_kpis, conn)
display(kpis)

,TIRAGEM_TOTAL,TOTAL_OPS,IMPRESSOES_INTERNAS,IMPRESSOES_EXTERNAS,OPS_MONOCAMADA,TIRAGEM_MEDIA_OP
0,14623772.57,117,59,12,46,124989.51


In [25]:
query_top5 = f"""
SELECT TOP 5
    NUOP,
    SUM(ISNULL(TIRAGEM, 0)) AS TIRAGEM_TOTAL
FROM dbo.AD_PCPCEN
WHERE DT_ENTREGA >= CONVERT(datetime, '{data_inicio}', 112)
  AND DT_ENTREGA <  CONVERT(datetime, '{data_fim}', 112)
GROUP BY NUOP
ORDER BY SUM(ISNULL(TIRAGEM, 0)) DESC;
"""

top5 = pd.read_sql(query_top5, conn)
display(top5)

,NUOP,TIRAGEM_TOTAL
0,58272,1904600.38
1,57784,830124.81
2,57665,774628.38
3,58260,774373.20
4,58273,672958.79


In [29]:
query_semana = f"""
SELECT
    DATEPART(WEEK, DT_ENTREGA) AS SEMANA,
    SUM(ISNULL(TIRAGEM, 0)) AS TIRAGEM_TOTAL
FROM dbo.AD_PCPCEN
WHERE DT_ENTREGA >= CONVERT(datetime, '{data_inicio}', 112)
  AND DT_ENTREGA <  CONVERT(datetime, '{data_fim}', 112)
GROUP BY DATEPART(WEEK, DT_ENTREGA)
ORDER BY SEMANA;
"""

semana = pd.read_sql(query_semana, conn)

display(semana)

,SEMANA,TIRAGEM_TOTAL
0,18,91375.56
1,19,5785968.97
2,20,631531.08
3,21,5525290.40
4,22,2589606.56


In [30]:
tiragem_total = float(kpis.loc[0, "TIRAGEM_TOTAL"] or 0)
total_ops = int(kpis.loc[0, "TOTAL_OPS"] or 0)
internas = int(kpis.loc[0, "IMPRESSOES_INTERNAS"] or 0)
externas = int(kpis.loc[0, "IMPRESSOES_EXTERNAS"] or 0)
monocamada = int(kpis.loc[0, "OPS_MONOCAMADA"] or 0)
tiragem_media = float(kpis.loc[0, "TIRAGEM_MEDIA_OP"] or 0)

semana["SEMANA"] = semana["SEMANA"].astype(str)

print("Tratamento OK")
print(tiragem_total, total_ops, internas, externas, monocamada, tiragem_media)

Tratamento OK
14623772.570000002 117 59 12 46 124989.51


In [34]:
fig = make_subplots(
    rows=3,
    cols=6,
    specs=[
        [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"},
         {"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],

        [{"type": "domain"}, {"type": "domain"}, {"type": "xy"},
         {"type": "xy"}, {"type": "xy"}, {"type": "xy"}],

        [{"type": "table"}, {"type": "table"}, {"type": "table"},
         {"type": "table"}, {"type": "table"}, {"type": "table"}],
    ],
    subplot_titles=[
        "", "", "", "", "", "",
        "Distribuição das OPs", "", "Tiragem por Semana", "", "", "",
        "Top 5 OPs por Tiragem", "", "", "", "", ""
    ],
    vertical_spacing=0.15,
    horizontal_spacing=0.06
)

# =========================
# KPIs
# =========================

kpi_config = [
    ("Tiragem Total", tiragem_total, "#1F77B4"),
    ("Total OPs", total_ops, "#0B1F3A"),
    ("Internas", internas, "#2CA02C"),
    ("Externas", externas, "#FF7F0E"),
    ("Monocamada", monocamada, "#9467BD"),
    ("Tiragem Média/OP", tiragem_media, "#606060"),
]

for i, (titulo, valor, cor) in enumerate(kpi_config, start=1):
    fig.add_trace(
        go.Indicator(
            mode="number",
            value=valor,
            number=dict(
                font=dict(size=30, color=cor),
                valueformat=",.0f"
            ),
            title=dict(
                text=f"<b>{titulo}</b>",
                font=dict(size=14)
            )
        ),
        row=1,
        col=i
    )

# =========================
# GRÁFICO ROSCA
# =========================

fig.add_trace(
    go.Pie(
        labels=["Internas", "Externas", "Monocamada"],
        values=[internas, externas, monocamada],
        hole=0.58,
        marker=dict(
            colors=["#2CA02C", "#FF7F0E", "#9467BD"]
        ),
        textinfo="label+percent"
    ),
    row=2,
    col=1
)

# =========================
# TIRAGEM POR SEMANA
# =========================

fig.add_trace(
    go.Bar(
        x=semana["SEMANA"],
        y=semana["TIRAGEM_TOTAL"],
        marker_color="#1F77B4",
        text=semana["TIRAGEM_TOTAL"].map(
            lambda x: f"{x:,.0f}".replace(",", ".")
        ),
        textposition="outside",
        name="Tiragem por Semana"
    ),
    row=2,
    col=3
)

# =========================
# TOP 5 OPS
# =========================

fig.add_trace(
    go.Table(
        header=dict(
            values=["<b>OP</b>", "<b>Tiragem Total</b>"],
            fill_color="#0B1F3A",
            font=dict(color="white", size=14),
            align="center",
            height=35
        ),
        cells=dict(
            values=[
                top5["NUOP"],
                top5["TIRAGEM_TOTAL"].map(
                    lambda x: f"{x:,.0f}".replace(",", ".")
                )
            ],
            fill_color="#F4F6F8",
            font=dict(color="#222222", size=13),
            align="center",
            height=30
        )
    ),
    row=3,
    col=1
)

# =========================
# LAYOUT
# =========================

fig.update_layout(
    title=dict(
        text="<b>Dashboard de Produção - Maio/2026</b>",
        x=0.5,
        xanchor="center",
        font=dict(
            size=24,
            color="#0B1F3A"
        )
    ),
    height=850,
    width=1350,
    paper_bgcolor="#F3F6FA",
    plot_bgcolor="white",
    showlegend=False,
    margin=dict(
        t=100,
        l=40,
        r=40,
        b=40
    )
)

fig.update_xaxes(
    title_text="Semana",
    row=2,
    col=3
)

fig.update_yaxes(
    title_text="Tiragem",
    tickformat=",.0f",
    showgrid=True,
    gridcolor="#E5EAF0",
    row=2,
    col=3
)

fig.show()

In [35]:
# =========================
# DASHBOARD VISUAL MELHORADO
# =========================

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "iframe_connected"

fig = make_subplots(
    rows=3,
    cols=12,
    specs=[
        [{"type": "indicator"}] * 12,
        [{"type": "domain"}] * 4 + [{"type": "xy"}] * 5 + [{"type": "table"}] * 3,
        [{"type": "xy"}] * 6 + [{"type": "table"}] * 6,
    ],
    subplot_titles=[
        "", "", "", "", "", "", "", "", "", "", "", "",
        "Distribuição das OPs", "", "", "",
        "Tiragem por Semana", "", "", "", "",
        "Top 5 OPs", "", "",
        "Resumo Semanal", "", "", "", "", "",
        "Ranking OPs por Tiragem", "", "", "", "", ""
    ],
    vertical_spacing=0.10,
    horizontal_spacing=0.035
)

# =========================
# KPIs
# =========================

kpis_cards = [
    ("Tiragem Total", tiragem_total, "#38BDF8"),
    ("Total OPs", total_ops, "#FFFFFF"),
    ("Internas", internas, "#22C55E"),
    ("Externas", externas, "#F97316"),
    ("Monocamada", monocamada, "#A855F7"),
    ("Média / OP", tiragem_media, "#FACC15"),
]

posicoes_kpis = [1, 3, 5, 7, 9, 11]

for col, (titulo, valor, cor) in zip(posicoes_kpis, kpis_cards):
    fig.add_trace(
        go.Indicator(
            mode="number",
            value=valor,
            number=dict(
                font=dict(size=26, color=cor),
                valueformat=",.0f"
            ),
            title=dict(
                text=f"<b>{titulo}</b>",
                font=dict(size=13, color="#CBD5E1")
            )
        ),
        row=1,
        col=col
    )

# =========================
# ROSCA OPs
# =========================

fig.add_trace(
    go.Pie(
        labels=["Internas", "Externas", "Monocamada"],
        values=[internas, externas, monocamada],
        hole=0.62,
        marker=dict(colors=["#22C55E", "#F97316", "#A855F7"]),
        textinfo="label+percent",
        textfont=dict(color="white", size=12),
        showlegend=False
    ),
    row=2,
    col=1
)

# =========================
# BARRAS SEMANAIS
# =========================

fig.add_trace(
    go.Bar(
        x=semana["SEMANA"],
        y=semana["TIRAGEM_TOTAL"],
        marker=dict(
            color=semana["TIRAGEM_TOTAL"],
            colorscale=[[0, "#1E3A8A"], [1, "#38BDF8"]],
            line=dict(color="#0F172A", width=1)
        ),
        text=semana["TIRAGEM_TOTAL"].map(lambda x: f"{x:,.0f}".replace(",", ".")),
        textposition="outside",
        name="Tiragem por Semana"
    ),
    row=2,
    col=5
)

# =========================
# TABELA TOP 5
# =========================

fig.add_trace(
    go.Table(
        header=dict(
            values=["<b>OP</b>", "<b>Tiragem</b>"],
            fill_color="#1E293B",
            font=dict(color="#F8FAFC", size=13),
            align="center",
            height=28
        ),
        cells=dict(
            values=[
                top5["NUOP"],
                top5["TIRAGEM_TOTAL"].map(lambda x: f"{x:,.0f}".replace(",", "."))
            ],
            fill_color="#0F172A",
            font=dict(color="#E2E8F0", size=12),
            align="center",
            height=26
        )
    ),
    row=2,
    col=10
)

# =========================
# LINHA / ÁREA SEMANAL
# =========================

fig.add_trace(
    go.Scatter(
        x=semana["SEMANA"],
        y=semana["TIRAGEM_TOTAL"],
        mode="lines+markers",
        line=dict(color="#22C55E", width=3),
        marker=dict(size=8, color="#22C55E"),
        fill="tozeroy",
        fillcolor="rgba(34,197,94,0.18)",
        name="Resumo Semanal"
    ),
    row=3,
    col=1
)

# =========================
# TABELA RANKING OPs
# =========================

fig.add_trace(
    go.Table(
        header=dict(
            values=["<b>#</b>", "<b>OP</b>", "<b>Tiragem Total</b>"],
            fill_color="#1E293B",
            font=dict(color="#F8FAFC", size=13),
            align="center",
            height=30
        ),
        cells=dict(
            values=[
                list(range(1, len(top5) + 1)),
                top5["NUOP"],
                top5["TIRAGEM_TOTAL"].map(lambda x: f"{x:,.0f}".replace(",", "."))
            ],
            fill_color="#0F172A",
            font=dict(color="#E2E8F0", size=12),
            align="center",
            height=27
        )
    ),
    row=3,
    col=7
)

# =========================
# EIXOS
# =========================

fig.update_xaxes(
    title_text="Semana",
    showgrid=False,
    color="#CBD5E1",
    row=2,
    col=5
)

fig.update_yaxes(
    title_text="Tiragem",
    tickformat=",.0f",
    gridcolor="#334155",
    color="#CBD5E1",
    row=2,
    col=5
)

fig.update_xaxes(
    title_text="Semana",
    showgrid=False,
    color="#CBD5E1",
    row=3,
    col=1
)

fig.update_yaxes(
    title_text="Tiragem",
    tickformat=",.0f",
    gridcolor="#334155",
    color="#CBD5E1",
    row=3,
    col=1
)

# =========================
# LAYOUT GERAL
# =========================

fig.update_layout(
    title=dict(
        text="<b>Dashboard de Produção - Maio/2026</b><br><sup>Indicadores gerais, distribuição das OPs, tiragem semanal e ranking</sup>",
        x=0.5,
        xanchor="center",
        font=dict(size=24, color="#F8FAFC")
    ),
    height=720,
    width=1280,
    paper_bgcolor="#020617",
    plot_bgcolor="#0F172A",
    font=dict(color="#E2E8F0"),
    showlegend=False,
    margin=dict(t=85, l=25, r=25, b=25)
)

for annotation in fig["layout"]["annotations"]:
    annotation["font"] = dict(size=14, color="#F8FAFC")

fig.show()

In [37]:
"""
================================================================================
  DASHBOARD DE PRODUÇÃO v2 — ESTILO POWER BI
  Gráficos em Canvas API nativa (zero dependências externas)
  Funciona 100% offline no Jupyter Notebook
================================================================================
"""

# ==============================================================================
# CÉLULA 1 — Imports e conexão
# ==============================================================================
import pandas as pd
import pyodbc
from IPython.display import display, HTML
import json

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=SANKHYA_MOCKUP;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)
print("Conexão OK")


# ==============================================================================
# CÉLULA 2 — Período
# ==============================================================================
data_inicio = "20260501"
data_fim    = "20260601"
print("Período:", data_inicio, "→", data_fim)


# ==============================================================================
# CÉLULA 3 — Queries SQL
# ==============================================================================
query_kpis = f"""
SELECT
    SUM(ISNULL(TIRAGEM, 0)) AS TIRAGEM_TOTAL,
    COUNT(DISTINCT NUOP)    AS TOTAL_OPS,
    SUM(CASE WHEN CODETAPA = 2 AND UPPER(ISNULL(TIPOIMPRESSAO,'')) LIKE 'I%' THEN 1 ELSE 0 END) AS IMPRESSOES_INTERNAS,
    SUM(CASE WHEN CODETAPA = 2 AND UPPER(ISNULL(TIPOIMPRESSAO,'')) LIKE 'E%' THEN 1 ELSE 0 END) AS IMPRESSOES_EXTERNAS,
    COUNT(DISTINCT NUOP)
    - (SUM(CASE WHEN CODETAPA = 2 AND UPPER(ISNULL(TIPOIMPRESSAO,'')) LIKE 'I%' THEN 1 ELSE 0 END)
      +SUM(CASE WHEN CODETAPA = 2 AND UPPER(ISNULL(TIPOIMPRESSAO,'')) LIKE 'E%' THEN 1 ELSE 0 END))
    AS OPS_MONOCAMADA,
    CAST(SUM(ISNULL(TIRAGEM,0)) / NULLIF(COUNT(DISTINCT NUOP),0) AS DECIMAL(18,2)) AS TIRAGEM_MEDIA_OP
FROM dbo.AD_PCPCEN
WHERE DT_ENTREGA >= CONVERT(datetime, '{data_inicio}', 112)
  AND DT_ENTREGA <  CONVERT(datetime, '{data_fim}', 112);
"""

query_top5 = f"""
SELECT TOP 5 NUOP, SUM(ISNULL(TIRAGEM, 0)) AS TIRAGEM_TOTAL
FROM dbo.AD_PCPCEN
WHERE DT_ENTREGA >= CONVERT(datetime, '{data_inicio}', 112)
  AND DT_ENTREGA <  CONVERT(datetime, '{data_fim}', 112)
GROUP BY NUOP ORDER BY SUM(ISNULL(TIRAGEM, 0)) DESC;
"""

query_semana = f"""
SELECT DATEPART(WEEK, DT_ENTREGA) AS SEMANA, SUM(ISNULL(TIRAGEM, 0)) AS TIRAGEM_TOTAL
FROM dbo.AD_PCPCEN
WHERE DT_ENTREGA >= CONVERT(datetime, '{data_inicio}', 112)
  AND DT_ENTREGA <  CONVERT(datetime, '{data_fim}', 112)
GROUP BY DATEPART(WEEK, DT_ENTREGA) ORDER BY SEMANA;
"""

kpis   = pd.read_sql(query_kpis,   conn)
top5   = pd.read_sql(query_top5,   conn)
semana = pd.read_sql(query_semana, conn)
print("Queries OK")


# ==============================================================================
# CÉLULA 4 — Tratamento
# ==============================================================================
tiragem_total = float(kpis.loc[0, "TIRAGEM_TOTAL"]        or 0)
total_ops     = int  (kpis.loc[0, "TOTAL_OPS"]            or 0)
internas      = int  (kpis.loc[0, "IMPRESSOES_INTERNAS"]  or 0)
externas      = int  (kpis.loc[0, "IMPRESSOES_EXTERNAS"]  or 0)
monocamada    = int  (kpis.loc[0, "OPS_MONOCAMADA"]       or 0)
tiragem_media = float(kpis.loc[0, "TIRAGEM_MEDIA_OP"]     or 0)

semana["SEMANA"] = semana["SEMANA"].astype(str)
semanas_labels   = semana["SEMANA"].tolist()
semanas_valores  = semana["TIRAGEM_TOTAL"].tolist()
top5_ops         = top5["NUOP"].astype(str).tolist()
top5_tiragens    = top5["TIRAGEM_TOTAL"].tolist()
print("Tratamento OK")


# ==============================================================================
# CÉLULA 5 — DASHBOARD
# ==============================================================================
dados_js = json.dumps({
    "tiragem_total":   tiragem_total,
    "total_ops":       total_ops,
    "internas":        internas,
    "externas":        externas,
    "monocamada":      monocamada,
    "tiragem_media":   tiragem_media,
    "semanas_labels":  semanas_labels,
    "semanas_valores": semanas_valores,
    "top5_ops":        top5_ops,
    "top5_tiragens":   top5_tiragens,
    "periodo":         f"{data_inicio[6:]}/{data_inicio[4:6]}/{data_inicio[:4]} até {data_fim[6:]}/{data_fim[4:6]}/{data_fim[:4]}",
})

html = f"""
<div id="dash-root" style="
  width:1280px; background:#0F172A; border-radius:12px;
  padding:22px; font-family:'Segoe UI',Arial,sans-serif; color:#E2E8F0;
  box-sizing:border-box;">

  <!-- ═══ HEADER ═══ -->
  <div style="display:flex;align-items:center;justify-content:space-between;
              border-bottom:1px solid #1E293B;padding-bottom:14px;margin-bottom:18px;">
    <div>
      <div style="font-size:20px;font-weight:700;color:#F8FAFC;">
        📊 Dashboard de Produção
      </div>
      <div id="hdr-per" style="font-size:12px;color:#64748B;margin-top:3px;"></div>
    </div>
    <div id="hdr-badge" style="background:#1E293B;border:1px solid #334155;
         border-radius:6px;padding:6px 14px;font-size:12px;color:#94A3B8;"></div>
  </div>

  <!-- ═══ KPIs ═══ -->
  <div style="display:grid;grid-template-columns:repeat(6,1fr);gap:12px;margin-bottom:16px;">
    <div class="kc" style="--a:#38BDF8">
      <div class="kl">Tiragem Total</div>
      <div class="kv" id="kv0" style="color:#38BDF8"></div>
      <div class="ks">unidades produzidas</div>
    </div>
    <div class="kc" style="--a:#FFFFFF">
      <div class="kl">Total de OPs</div>
      <div class="kv" id="kv1" style="color:#FFFFFF"></div>
      <div class="ks">ordens de produção</div>
    </div>
    <div class="kc" style="--a:#22C55E">
      <div class="kl">Impressões Internas</div>
      <div class="kv" id="kv2" style="color:#22C55E"></div>
      <div class="ks" id="ks2"></div>
    </div>
    <div class="kc" style="--a:#F97316">
      <div class="kl">Impressões Externas</div>
      <div class="kv" id="kv3" style="color:#F97316"></div>
      <div class="ks" id="ks3"></div>
    </div>
    <div class="kc" style="--a:#A855F7">
      <div class="kl">Monocamada</div>
      <div class="kv" id="kv4" style="color:#A855F7"></div>
      <div class="ks" id="ks4"></div>
    </div>
    <div class="kc" style="--a:#FACC15">
      <div class="kl">Tiragem Média / OP</div>
      <div class="kv" id="kv5" style="color:#FACC15"></div>
      <div class="ks">média por ordem</div>
    </div>
  </div>

  <!-- ═══ LINHA MEIO ═══ -->
  <div style="display:grid;grid-template-columns:240px 1fr 280px;gap:12px;margin-bottom:16px;">

    <!-- Rosca -->
    <div class="card">
      <div class="ct" style="--a:#38BDF8">Distribuição das OPs</div>
      <canvas id="cvRosca" width="208" height="208"
        style="display:block;margin:0 auto 10px"></canvas>
      <div id="leg-rosca"></div>
    </div>

    <!-- Barras semanais -->
    <div class="card">
      <div class="ct" style="--a:#38BDF8">Tiragem por Semana</div>
      <canvas id="cvBarras" width="680" height="220"
        style="display:block;width:100%;height:220px"></canvas>
    </div>

    <!-- Top 5 -->
    <div class="card">
      <div class="ct" style="--a:#FACC15">Top 5 OPs · Tiragem</div>
      <table style="width:100%;border-collapse:collapse;font-size:12px;">
        <thead>
          <tr>
            <th style="width:28px;padding:6px 8px;color:#64748B;font-size:10px;
                text-transform:uppercase;border-bottom:1px solid #334155;">#</th>
            <th style="padding:6px 8px;color:#64748B;font-size:10px;
                text-transform:uppercase;border-bottom:1px solid #334155;text-align:left">OP</th>
            <th style="padding:6px 8px;color:#64748B;font-size:10px;
                text-transform:uppercase;border-bottom:1px solid #334155;text-align:right">Tiragem</th>
          </tr>
        </thead>
        <tbody id="top5-body"></tbody>
      </table>
    </div>

  </div>

  <!-- ═══ LINHA BAIXO ═══ -->
  <div style="display:grid;grid-template-columns:1fr 280px;gap:12px;">

    <!-- Área semanal -->
    <div class="card">
      <div class="ct" style="--a:#22C55E">Evolução Semanal da Tiragem</div>
      <canvas id="cvArea" width="940" height="155"
        style="display:block;width:100%;height:155px"></canvas>
    </div>

    <!-- Tabela semanal -->
    <div class="card">
      <div class="ct" style="--a:#38BDF8">Resumo por Semana</div>
      <table style="width:100%;border-collapse:collapse;font-size:12px;">
        <thead>
          <tr>
            <th style="padding:6px 8px;color:#64748B;font-size:10px;text-transform:uppercase;
                border-bottom:1px solid #334155;text-align:left">Semana</th>
            <th style="padding:6px 8px;color:#64748B;font-size:10px;text-transform:uppercase;
                border-bottom:1px solid #334155;text-align:right">Tiragem</th>
            <th style="padding:6px 8px;color:#64748B;font-size:10px;text-transform:uppercase;
                border-bottom:1px solid #334155;text-align:right">%</th>
          </tr>
        </thead>
        <tbody id="sem-body"></tbody>
      </table>
    </div>

  </div>
</div>

<style>
  .kc {{ background:#1E293B; border-radius:8px; padding:14px 16px;
        border-left:4px solid var(--a); }}
  .kl {{ font-size:10px; color:#64748B; text-transform:uppercase;
        letter-spacing:.8px; margin-bottom:6px; }}
  .kv {{ font-size:26px; font-weight:700; line-height:1; }}
  .ks {{ font-size:11px; color:#475569; margin-top:5px; }}
  .card {{ background:#1E293B; border-radius:8px; padding:16px; }}
  .ct {{ font-size:11px; font-weight:600; color:#94A3B8; text-transform:uppercase;
        letter-spacing:.8px; margin-bottom:12px; display:flex; align-items:center; gap:7px; }}
  .ct::before {{ content:''; display:inline-block; width:3px; height:13px;
                background:var(--a,#38BDF8); border-radius:2px; }}
</style>

<script>
(function() {{
  var D = {dados_js};

  // ── Formatadores ────────────────────────────────────────────────
  function fmt(v) {{
    return Number(v).toLocaleString('pt-BR', {{maximumFractionDigits:0}});
  }}
  function fmtK(v) {{
    if (v >= 1e6) return (v/1e6).toFixed(1) + 'M';
    if (v >= 1e3) return (v/1e3).toFixed(1) + 'K';
    return fmt(v);
  }}
  function pct(v, t) {{
    return t ? (v/t*100).toFixed(1) + '%' : '—';
  }}

  // ── Header ──────────────────────────────────────────────────────
  document.getElementById('hdr-per').textContent   = 'Período: ' + D.periodo;
  document.getElementById('hdr-badge').textContent = 'Total: ' + fmt(D.tiragem_total) + ' un.';

  // ── KPIs ────────────────────────────────────────────────────────
  document.getElementById('kv0').textContent = fmtK(D.tiragem_total);
  document.getElementById('kv1').textContent = D.total_ops;
  document.getElementById('kv2').textContent = D.internas;
  document.getElementById('kv3').textContent = D.externas;
  document.getElementById('kv4').textContent = D.monocamada;
  document.getElementById('kv5').textContent = fmtK(D.tiragem_media);
  document.getElementById('ks2').textContent = pct(D.internas,  D.total_ops) + ' das OPs';
  document.getElementById('ks3').textContent = pct(D.externas,  D.total_ops) + ' das OPs';
  document.getElementById('ks4').textContent = pct(D.monocamada,D.total_ops) + ' das OPs';

  // ── ROSCA (Canvas API puro) ─────────────────────────────────────
  var cvR = document.getElementById('cvRosca');
  var ctR = cvR.getContext('2d');
  var cx = 104, cy = 104, ro = 92, ri = 60;
  var vals   = [D.internas, D.externas, D.monocamada];
  var cores  = ['#22C55E','#F97316','#A855F7'];
  var labels = ['Internas','Externas','Monocamada'];
  var total  = vals.reduce(function(a,b){{return a+b;}}, 0) || 1;
  var ang = -Math.PI / 2;
  vals.forEach(function(v, i) {{
    var slice = (v / total) * 2 * Math.PI;
    ctR.beginPath();
    ctR.moveTo(cx, cy);
    ctR.arc(cx, cy, ro, ang, ang + slice);
    ctR.closePath();
    ctR.fillStyle = cores[i];
    ctR.fill();
    // furo
    ctR.beginPath();
    ctR.arc(cx, cy, ri, 0, 2*Math.PI);
    ctR.fillStyle = '#1E293B';
    ctR.fill();
    ang += slice;
  }});
  // gap branco entre fatias
  ang = -Math.PI / 2;
  vals.forEach(function(v) {{
    var slice = (v / total) * 2 * Math.PI;
    ctR.beginPath();
    ctR.moveTo(cx + Math.cos(ang)*ri, cy + Math.sin(ang)*ri);
    ctR.lineTo(cx + Math.cos(ang)*ro, cy + Math.sin(ang)*ro);
    ctR.strokeStyle = '#0F172A';
    ctR.lineWidth = 2;
    ctR.stroke();
    ang += slice;
  }});

  // Legenda rosca
  var legEl = document.getElementById('leg-rosca');
  labels.forEach(function(l, i) {{
    legEl.innerHTML += '<div style="display:flex;align-items:center;gap:8px;'
      +'font-size:12px;color:#CBD5E1;margin-bottom:6px;">'
      +'<span style="width:9px;height:9px;border-radius:50%;background:'+cores[i]
      +';flex-shrink:0;display:inline-block"></span>'
      +'<span>'+l+'</span>'
      +'<span style="margin-left:auto;color:#64748B">'+vals[i]+' &nbsp;'+pct(vals[i],D.total_ops)+'</span>'
      +'</div>';
  }});

  // ── BARRAS (Canvas API puro) ────────────────────────────────────
  var cvB = document.getElementById('cvBarras');
  // canvas com dimensões reais
  var W = cvB.offsetWidth  || 680;
  var H = 220;
  cvB.width  = W;
  cvB.height = H;
  var ctB = cvB.getContext('2d');
  var pad = {{t:20, r:20, b:35, l:65}};
  var n   = D.semanas_valores.length;
  var maxV = Math.max.apply(null, D.semanas_valores) * 1.15 || 1;
  var barW = Math.floor((W - pad.l - pad.r) / n * 0.6);
  var gap  = (W - pad.l - pad.r) / n;

  // grid horizontal
  var steps = 4;
  for (var s = 0; s <= steps; s++) {{
    var yg = pad.t + (H - pad.t - pad.b) * (1 - s/steps);
    ctB.beginPath();
    ctB.moveTo(pad.l, yg); ctB.lineTo(W - pad.r, yg);
    ctB.strokeStyle = '#1E293B'; ctB.lineWidth = 1; ctB.stroke();
    var lbl = fmtK(maxV / 1.15 * s / steps);
    ctB.fillStyle = '#475569'; ctB.font = '10px Segoe UI';
    ctB.textAlign = 'right';
    ctB.fillText(lbl, pad.l - 6, yg + 3);
  }}

  // barras
  D.semanas_valores.forEach(function(v, i) {{
    var x  = pad.l + gap * i + (gap - barW) / 2;
    var bh = (v / maxV) * (H - pad.t - pad.b);
    var y  = H - pad.b - bh;
    // gradiente por intensidade
    var alpha = (0.4 + 0.6 * v / Math.max.apply(null, D.semanas_valores)).toFixed(2);
    ctB.fillStyle = 'rgba(56,189,248,' + alpha + ')';
    ctB.beginPath();
    ctB.roundRect ? ctB.roundRect(x, y, barW, bh, [4,4,0,0])
                  : ctB.rect(x, y, barW, bh);
    ctB.fill();
    // label no topo
    ctB.fillStyle = '#94A3B8'; ctB.font = '10px Segoe UI';
    ctB.textAlign = 'center';
    ctB.fillText(fmtK(v), x + barW/2, y - 5);
    // label eixo X
    ctB.fillStyle = '#64748B'; ctB.font = '11px Segoe UI';
    ctB.fillText('Sem ' + D.semanas_labels[i], x + barW/2, H - pad.b + 14);
  }});

  // ── ÁREA (Canvas API puro) ──────────────────────────────────────
  var cvA = document.getElementById('cvArea');
  var WA = cvA.offsetWidth || 940;
  cvA.width = WA; cvA.height = 155;
  var ctA = cvA.getContext('2d');
  var padA = {{t:18, r:20, b:30, l:65}};
  var maxA = Math.max.apply(null, D.semanas_valores) * 1.2 || 1;
  var pts  = D.semanas_valores.map(function(v, i) {{
    return {{
      x: padA.l + i * (WA - padA.l - padA.r) / (n > 1 ? n-1 : 1),
      y: padA.t + (155 - padA.t - padA.b) * (1 - v / maxA)
    }};
  }});

  // grid
  for (var s2 = 0; s2 <= 3; s2++) {{
    var yg2 = padA.t + (155 - padA.t - padA.b) * (1 - s2/3);
    ctA.beginPath();
    ctA.moveTo(padA.l, yg2); ctA.lineTo(WA - padA.r, yg2);
    ctA.strokeStyle = '#1E293B'; ctA.lineWidth = 1; ctA.stroke();
    ctA.fillStyle = '#475569'; ctA.font = '10px Segoe UI'; ctA.textAlign = 'right';
    ctA.fillText(fmtK(maxA / 1.2 * s2 / 3), padA.l - 6, yg2 + 3);
  }}

  if (pts.length > 1) {{
    // área preenchida
    var grad = ctA.createLinearGradient(0, padA.t, 0, 155 - padA.b);
    grad.addColorStop(0,   'rgba(34,197,94,0.25)');
    grad.addColorStop(1,   'rgba(34,197,94,0)');
    ctA.beginPath();
    ctA.moveTo(pts[0].x, 155 - padA.b);
    ctA.lineTo(pts[0].x, pts[0].y);
    for (var i2 = 1; i2 < pts.length; i2++) {{
      var cp1x = (pts[i2-1].x + pts[i2].x) / 2;
      ctA.bezierCurveTo(cp1x, pts[i2-1].y, cp1x, pts[i2].y, pts[i2].x, pts[i2].y);
    }}
    ctA.lineTo(pts[pts.length-1].x, 155 - padA.b);
    ctA.closePath();
    ctA.fillStyle = grad;
    ctA.fill();

    // linha
    ctA.beginPath();
    ctA.moveTo(pts[0].x, pts[0].y);
    for (var i3 = 1; i3 < pts.length; i3++) {{
      var cp2x = (pts[i3-1].x + pts[i3].x) / 2;
      ctA.bezierCurveTo(cp2x, pts[i3-1].y, cp2x, pts[i3].y, pts[i3].x, pts[i3].y);
    }}
    ctA.strokeStyle = '#22C55E'; ctA.lineWidth = 2.5; ctA.stroke();

    // pontos
    pts.forEach(function(p, i4) {{
      ctA.beginPath();
      ctA.arc(p.x, p.y, 4, 0, 2*Math.PI);
      ctA.fillStyle = '#22C55E'; ctA.fill();
      ctA.strokeStyle = '#0F172A'; ctA.lineWidth = 2; ctA.stroke();
      // label X
      ctA.fillStyle = '#64748B'; ctA.font = '10px Segoe UI'; ctA.textAlign = 'center';
      ctA.fillText('S' + D.semanas_labels[i4], p.x, 155 - padA.b + 14);
    }});
  }}

  // ── TOP 5 tabela ────────────────────────────────────────────────
  var maxT = Math.max.apply(null, D.top5_tiragens) || 1;
  var tbody = document.getElementById('top5-body');
  D.top5_ops.forEach(function(op, i) {{
    var t = D.top5_tiragens[i];
    var bw = (t / maxT * 100).toFixed(0);
    tbody.innerHTML += '<tr style="border-bottom:1px solid #1E293B">'
      +'<td style="padding:8px 8px;color:#38BDF8;font-weight:700;font-size:11px">'+ (i+1) +'</td>'
      +'<td style="padding:8px 8px">'
      +'  <div style="font-weight:600;font-size:12px;color:#F1F5F9">'+op+'</div>'
      +'  <div style="height:3px;background:#0F172A;border-radius:2px;margin-top:5px">'
      +'    <div style="height:3px;width:'+bw+'%;background:#FACC15;border-radius:2px"></div>'
      +'  </div>'
      +'</td>'
      +'<td style="padding:8px 8px;text-align:right;color:#94A3B8;font-size:12px">'+fmtK(t)+'</td>'
      +'</tr>';
  }});

  // ── Tabela semanal ───────────────────────────────────────────────
  var totalSem = D.semanas_valores.reduce(function(a,b){{return a+b;}},0);
  var semBd = document.getElementById('sem-body');
  D.semanas_labels.forEach(function(s3, i) {{
    var v = D.semanas_valores[i];
    semBd.innerHTML += '<tr style="border-bottom:1px solid #1E293B">'
      +'<td style="padding:7px 8px;color:#E2E8F0;font-size:12px">Semana '+s3+'</td>'
      +'<td style="padding:7px 8px;text-align:right;color:#94A3B8;font-size:12px">'+fmtK(v)+'</td>'
      +'<td style="padding:7px 8px;text-align:right;color:#38BDF8;font-size:12px">'+pct(v,totalSem)+'</td>'
      +'</tr>';
  }});

}})();
</script>
"""

display(HTML(html))

Conexão OK
Período: 20260501 → 20260601
Queries OK
Tratamento OK


#,OP,Tiragem
Semana,Tiragem,%
